# Customizing agent memory


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# import the tool function and system prompt from the lib
from lib.order_process_helper import get_order_status, CUSTOMER_SERVICE_ASSISTANT_SYSTEM_PROMPT

In [3]:
# 繼承 `AgentState`，加入自訂欄位

from langchain.agents import AgentState

class OrderAgentState(AgentState):
    user_id: str

In [4]:
# Create the InMemorySaver checkpointer 
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [5]:
# Create the agent
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_order_status],
    system_prompt=CUSTOMER_SERVICE_ASSISTANT_SYSTEM_PROMPT,
    checkpointer=checkpointer,
    # 使用自訂的 AgentState
    state_schema=OrderAgentState  
)

In [6]:
# Create the initial agent state with the user_id


# 假設這是登入系統完成驗證後回傳的結果
from langchain.messages import HumanMessage

authenticated_user_id = "user_123"

# Human message
human_message = HumanMessage("請查詢訂單 A1024 的狀態。")

# the initial agent state 
# AgentState 是一個 TypedDict，所以 OrderAgentState 也是一個 TypedDict
# 所以可直接使用字典的方式建立初始狀態
init_state = {
    "messages": [human_message],
    "user_id": authenticated_user_id
}

In [7]:
# Define the RunnableConfig to set the thread_id 
# 使用建構函數
from langchain_core.runnables.config import RunnableConfig

runnable_config = RunnableConfig(
    configurable={
        "thread_id": "thread_123",
    }
)
# 相當於直接使用字典的方式建立
# runnable_config = {
#     "configurable": {
#          "thread_id": "thread_123"
#     }
# }

In [8]:
# 第一次 invoke 
response_1 = agent.invoke(
    input=init_state,
    config=runnable_config
)

In [11]:
response_1['user_id']

'user_123'

In [11]:
from pprint import pp 
pp(response_1["messages"][-1].content)

'訂單 A1024 的目前狀態為：已出貨，預計送達日期為 2023-07-25。需要我提供追蹤碼或更多詳情嗎？'


In [16]:
# 查看 state 中的 user_id
pp(response_1["user_id"])

'user_123'
